# Notebook 4 — Trust but Verify

**Goal:** don't just consume the API's precomputed numbers — reproduce one of them
yourself from raw geometry, and figure out why it might (or might not) match.

The API docs for `/risk/features/multi.geojson` say the geometry it serves comes
from **a precomputed cache** — *"GeoJSON simplified to 1m tolerance, clipped to dam
zones during import"* — built for fast map rendering. `/risk/summary`'s counts, on
the other hand, are described as coming from the canonical risk metrics pipeline.
Are they counting the same thing? Let's check, instead of assuming.


In [ ]:
import requests
import pandas as pd
import geopandas as gpd

BASE_URL = "http://149.165.154.170:30080"
DAM = "UT00221"  # Mountain Dell
TARGETS = ["hospitals", "railroads", "svi_tracts", "gap_status", "transportation"]


def get_json(path, **params):
    resp = requests.get(f"{BASE_URL}{path}", params=params, timeout=20)
    resp.raise_for_status()
    return resp.json()


summary = get_json("/risk/summary", damnumber=DAM, targets=TARGETS)
precomputed = summary["counts"]
precomputed


## Recomputing from raw geometry

For each target: pull the dam's zone geometry and the target's feature geometry
straight from the API, then do our *own* spatial join with `geopandas.sjoin` and
count the matches — no trusting the API's arithmetic, just its geometry.


In [ ]:
zone_gdf = gpd.GeoDataFrame.from_features(
    get_json("/risk/zone.geojson", damnumber=DAM)["features"], crs="EPSG:4326"
)

rows = []
for target in TARGETS:
    feats = get_json("/risk/features/multi.geojson", damnumber=DAM, targets=target)["features"]
    target_gdf = gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326")
    joined = gpd.sjoin(target_gdf, zone_gdf, predicate="intersects", how="inner")
    rows.append({"target": target, "precomputed": precomputed[target], "self_joined": len(joined)})

comparison = pd.DataFrame(rows)
comparison["match"] = comparison["precomputed"] == comparison["self_joined"]
comparison


## Reading the result

Four out of five targets match exactly — a good sanity check that the cached
geometry is trustworthy for counting points, lines, and (in this case) census-tract
polygons. But **`gap_status` doesn't match**: the precomputed summary says 51,
our spatial join says 1. That's not a rounding error — something structural is
different. Let's look at the raw `gap_status` features directly.


In [ ]:
gap_feats = get_json("/risk/features/multi.geojson", damnumber=DAM, targets="gap_status")["features"]
print("raw feature count returned by the cache:", len(gap_feats))
gap_feats[0]["properties"]


## The explanation

There's really only one polygon feature for `gap_status` in the cache — and its
properties include a `count_objectid` field, a hint that several original protected-
area parcels were **dissolved into one merged polygon** before being cached (a
common trick to keep display GeoJSON small and fast to render). The precomputed
`/risk/summary` count of 51, by contrast, appears to count the original underlying
parcels *before* that dissolve.

In other words: `/risk/features/multi.geojson` is a **display cache** optimized for
drawing a map quickly, not a source-of-truth record count. `/risk/summary` and
`/risk/metrics` are the ones counting original rows. For point layers (hospitals) and
already-non-overlapping polygons (census tracts, which don't get merged since they
don't overlap each other) the two agree — but for `gap_status`, where overlapping
ownership parcels get unioned together, a naive "just count what's on the map"
approach silently undercounts by a lot.

**Lesson:** a feature count from a rendering-optimized geometry layer is not
automatically the same thing as the count backing a summary statistic — even when
both come from the same API. Always check what a cache was optimized for before
treating its shape as ground truth.


## Exercise

We saw `svi_tracts` (29 vs. 29) stay consistent while `gap_status` (51 vs. 1) did
not. Both are polygon targets, so polygon-vs-point isn't the deciding factor. Pull
the raw `svi_tracts` features (`targets="svi_tracts"`) the same way we did for
`gap_status` above and inspect the feature count and properties. Why do you think
census tracts *don't* get dissolved the way protected-area parcels do? (Hint: think
about whether the source polygons for each target overlap one another.)


In [ ]:
# your investigation here
